## Data Quality Analysis (Missing, Duplicates and Data Types fixes)

In [2]:
import pandas as pd
import numpy as np
from configs.settings import project_dir

In [3]:
p_dir = project_dir()

#path to raw data files
raw_data_dir = p_dir.RAW_DATA_DIR
customers = raw_data_dir/'olist_customers_dataset.csv'
location = raw_data_dir/'olist_geolocation_dataset.csv'
items = raw_data_dir/'olist_order_items_dataset.csv'
payments = raw_data_dir/'olist_order_payments_dataset.csv'
reviews = raw_data_dir/'olist_order_reviews_dataset.csv'
orders = raw_data_dir/'olist_orders_dataset.csv'
products = raw_data_dir/'olist_products_dataset.csv'
sellers = raw_data_dir/'olist_sellers_dataset.csv'
category = raw_data_dir /'product_category_name_translation.csv'

In [4]:
# clean data dir
clean_data_dir = p_dir.CLEANED_DIR

### Customers Dataset 

In [5]:
customers_df = pd.read_csv(customers)
customers_df.sample(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
44321,5e085412648734bfe73b0c0196075f49,2f5bd6ac7c9fe91b6667f78c775c880e,31140,belo horizonte,MG
52367,0462be0a9892418817ec41fd7c39a57b,db70a8a87fd3ae2ec9b0d9e9daada859,37650,camanducaia,MG
78204,80982a05bcad06c7ff17a85f5684d05a,b4e59ceaee1a495eb05d5ed677cc0783,13054,campinas,SP
16206,ad33089f3dc319f48cc1877a5d4f8a48,df924fa71a293b65069016d31cd28a2e,99714,erechim,RS
74914,a6e8240939fddfd24161a340c6589aaf,01ac1e0b4421bb9946e56692e21226f0,24120,niteroi,RJ


In [6]:
# Missing Values
print(customers_df.isnull().sum())
print(f"\n\nNumber of rows with duplicates rows: {customers_df.duplicated().sum()}\n\n")
customers_df.info()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


Number of rows with duplicates rows: 0


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [7]:
customers_df['customer_city'].value_counts().to_string('../arson/customers_city.csv')

In [8]:
print(f"Number of duplicates value in customer_city col is {customers_df['customer_city'].unique().duplicated().sum()}\n")
print(f"Number of duplicates values in customer state is {customers_df['customer_state'].value_counts().duplicated().sum()}")

customers_df['customer_zip_code_prefix'].value_counts().sort_values(ascending=False).to_string('../arson/zip_code.txt')

Number of duplicates value in customer_city col is 0

Number of duplicates values in customer state is 0


In [9]:
customers_df.to_csv(clean_data_dir/'clean_customers.csv')

### Geolocation Dataset

In [10]:
location_df = pd.read_csv(location)
location_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [11]:
location_df.shape

(1000163, 5)

In [12]:
# print(location_df.isnull().sum())
# print(f"Number of Duplicates in location dataset of {location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city.txt')}")

location_df['geolocation_city'] = location_df['geolocation_city'].astype('string')
location_df['geolocation_city'] = location_df['geolocation_city'].str.strip()
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('`','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('^','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('~','',regex='False')
# location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city_clean.txt')

In [13]:
location_df[['geolocation_lat','geolocation_lng']].duplicated().sum()

np.int64(281700)

In [14]:
location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/lat.csv')


In [15]:
location_df.head()
location_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  string 
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(1), string(1)
memory usage: 38.2 MB


In [16]:
messy_data = location_df[location_df[['geolocation_lat','geolocation_lng']].duplicated()]
# messy_data[['geolocation_city','geolocation_lat','geolocation_lng']].value_counts().head(30)
messy_data['geolocation_city'] = messy_data['geolocation_city'].astype(str)
messy_data['geolocation_city'] = messy_data['geolocation_city'].str.strip()
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].groupby(['geolocation_lat','geolocation_lng']).value_counts().sort_values(ascending=False).to_string('../arson/messy1.csv')

In [17]:
messy_data[['geolocation_lat','geolocation_lng']].duplicated()

15         False
44          True
65          True
66         False
67          True
           ...  
1000153     True
1000154     True
1000159    False
1000160    False
1000162     True
Length: 281700, dtype: bool

In [18]:
messy_data[messy_data['geolocation_city'] == 'xambre']

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
897964,87535,-23.729684,-53.489767,xambre,PR
898303,87535,-23.729684,-53.489767,xambre,PR
898430,87535,-23.733100,-53.488682,xambre,PR
898450,87535,-23.737584,-53.485975,xambre,PR


In [19]:
messy_data[['geolocation_city','geolocation_lat','geolocation_lng']][messy_data[['geolocation_lat','geolocation_lng']].duplicated()].head(30).sort_values(by=['geolocation_lat','geolocation_lng'])

,geolocation_city,geolocation_lat,geolocation_lng
237,sao paulo,-23.552496,-46.632060
337,sao paulo,-23.552235,-46.628441
136,sao paulo,-23.549854,-46.643139
161,sao paulo,-23.549854,-46.643139
223,sao paulo,-23.549854,-46.643139
240,sao paulo,-23.549854,-46.643139
280,sao paulo,-23.549819,-46.635606
253,sao paulo,-23.546935,-46.636588
275,sao paulo,-23.546935,-46.636588
306,são paulo,-23.546935,-46.636588


In [20]:
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].where(messy_data[['geolocation_lat','geolocation_lng']].duplicated())

,geolocation_lat,geolocation_lng,geolocation_city
15,NaN,NaN,NaN
44,-23.546081,-46.644820,sao paulo
65,-23.546081,-46.644820,sao paulo
66,NaN,NaN,NaN
67,-23.546081,-46.644820,sao paulo
...,...,...,...
1000153,-28.343273,-51.873734,ciriaco
1000154,-28.070493,-52.011342,tapejara
1000159,NaN,NaN,NaN
1000160,NaN,NaN,NaN


In [21]:
messy_data[messy_data['geolocation_zip_code_prefix'].duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
...,...,...,...,...,...
1000153,99970,-28.343273,-51.873734,ciriaco,RS
1000154,99950,-28.070493,-52.011342,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS


In [22]:
location_df[location_df['geolocation_zip_code_prefix'].duplicated()].sort_values(by='geolocation_zip_code_prefix').head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1384,1001,-23.549292,-46.633559,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
1351,1001,-23.549951,-46.634027,são paulo,SP
235,1001,-23.550642,-46.634410,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP
1004,1001,-23.549292,-46.633559,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
519,1001,-23.551337,-46.634027,sao paulo,SP
1062,1001,-23.550498,-46.634338,sao paulo,SP
299,1001,-23.549698,-46.633909,sao paulo,SP


In [23]:
location_df_clean = location_df.drop_duplicates(subset=['geolocation_lat','geolocation_lng'])

In [24]:
print(location_df.shape)
print(location_df_clean.shape)

(1000163, 5)
(718463, 5)


In [25]:
1000163 - 718463

281700

In [26]:
location_df.duplicated().sum()

np.int64(261831)

In [27]:
location_df_clean.duplicated().sum()

np.int64(0)

In [28]:
location_df_clean.columns

Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')

In [29]:
location_df_clean = location_df_clean.rename(columns={
    'geolocation_zip_code_prefix' : 'zip_code',
    'geolocation_lat' : 'lat',
    'geolocation_lng' : 'lng',
    'geolocation_city' : 'city',
    'geolocation_state' : 'state'
})

In [30]:
# Saving cleaned dataset
location_df_clean.to_csv(clean_data_dir/'cleaned_location.csv')

In [31]:
location_df_clean['zip_code'].duplicated().sum()

np.int64(699523)

## Order Items dataset cleaning

In [32]:
order_items_df = pd.read_csv(items)
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [33]:
order_items_df['order_item_id'].value_counts()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [34]:
order_items_df.info()
# No null values, only incorrect dtype of date column

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [35]:
order_items_df['order_item_id'].value_counts()
# order_items_df.duplicated().sum()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [36]:
# fixing the dtype of date col
order_items_df['shipping_limit_date'] = order_items_df['shipping_limit_date'].astype('datetime64[ns]')

In [37]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [38]:
order_items_df.to_csv(clean_data_dir/'cleaned_orders_items.csv')

### Order Payments Dataset cleaning

In [39]:
order_payments_df = pd.read_csv(payments)
order_payments_df.sample(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
75288,833124c58a852f0accaa7c09219e2967,1,voucher,1,35.00
16993,2045198a6e7202ea847fd4816f7dde74,1,credit_card,3,128.80
55150,51ba136bb458846fcc5c1f745da00398,1,credit_card,4,44.32
92337,431fe12f4d24d8f61a0947c0bc842a04,1,credit_card,2,63.10
66154,0e94db37ea63222e95092a0f870f8e9a,1,credit_card,6,63.27
40810,31dc5103a73f31b15f42c3a5367efcf0,1,credit_card,5,130.46
2378,7d5c1320ac55581cd5de4d8f83c99731,1,boleto,1,166.59
80444,a3ba7fa5266d62188949da3d6ccc3884,1,credit_card,8,133.58
88832,591398abee6f2509ff5cdb6e5068dc27,1,credit_card,10,289.95
39957,e9e702d3e92379f53b4805d98c094bd7,1,credit_card,2,171.94


In [40]:
# order_payments_df.info()
# order_payments_df['payment_type'].value_counts()
order_payments_df.groupby('payment_type')['payment_installments'].value_counts()

payment_type  payment_installments
boleto        1                       19784
credit_card   1                       25455
              2                       12413
              3                       10461
              4                        7098
              10                       5328
              5                        5239
              8                        4268
              6                        3920
              7                        1626
              9                         644
              12                        133
              15                         74
              18                         27
              11                         23
              24                         18
              20                         17
              13                         16
              14                         15
              17                          8
              16                          5
              21                         

In [41]:
order_payments_df.duplicated().sum()

np.int64(0)

In [42]:
## replace boleto with wallet, handled not_defined values with median
order_payments_df['payment_type'] = order_payments_df['payment_type'].str.replace('boleto','wallet')

order_payments_df['payment_type'] = order_payments_df['payment_type'].str.replace('not_defined',order_payments_df['payment_type'].mode()[0])

In [43]:
order_payments_df['payment_type'].value_counts()

payment_type
credit_card    76798
wallet         19784
voucher         5775
debit_card      1529
Name: count, dtype: int64

In [44]:
order_payments_df.to_csv(clean_data_dir/'cleaned_order_payments.csv')

## Order Reviews Dataset cleaning

In [45]:
order_reviews_df = pd.read_csv(reviews)
order_reviews_df.sample(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7233,6f02b79f51bf00f236a17adea1dd74e0,d827e1e47dc185687102015449ff2e4a,3,NaN,NaN,2018-03-30 00:00:00,2018-04-02 02:30:35
80972,2fe1c143280e646e97b2b90861d90d5f,f8c830660882d73b51a825170d8ef567,3,NaN,NaN,2017-08-29 00:00:00,2017-08-29 23:01:42
2347,048314354fefee6073f26477c739665d,73f4e07e698c5c4442249571348109f1,5,NaN,NaN,2018-02-17 00:00:00,2018-02-19 13:39:55
56855,a62ea544fbc9576b646ab34058656791,14419beee19b41341bead9fe7fc49d76,5,NaN,NaN,2017-10-26 00:00:00,2017-10-27 02:45:24
11669,30abe4d392bc9d3ab08afae6383e3853,49f0bd432488ec2cc7a18c0c786b743f,5,NaN,Entregue muito rápido\r\nParabéns,2017-12-14 00:00:00,2017-12-14 20:09:47
53032,7b6720ed19fdda937039c53eee92e13c,19938b0f4dd0796e7c2d07f8a406add7,5,NaN,NaN,2017-02-21 00:00:00,2017-02-22 11:21:08
2608,d5112e1a5ffc63506fb53cce596250ef,3832a9766706c5e6bbfa5d101f782873,5,NaN,NaN,2017-10-31 00:00:00,2017-10-31 15:00:50
47699,2f59a88f994c5d6f802f0ddab0dd785b,e74059734326eb12073dc5cd441fccb6,1,Produto não entregue,O produto ainda não foi entregue. Recebi a men...,2018-07-15 00:00:00,2018-07-16 14:38:59
16816,6bd9d1dd03fbefc04b0a40fe4bf7c7ce,1dda4cf38de17d1b18e9132ed692d064,5,NaN,NaN,2017-05-26 00:00:00,2017-05-26 23:33:21
39,9fd59cd04b42f600df9f25e54082a8d1,3c314f50bc654f3c4e317b055681dff9,1,NaN,Nada de chegar o meu pedido.,2017-04-21 00:00:00,2017-04-23 05:37:03


In [46]:
order_reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [47]:
order_reviews_df.duplicated().sum()
order_reviews_df['review_score'].value_counts()
# TO-DO
## handle the date dtype col, handle missing value in both title and comment message.(a better approach to handle, cant drop all these rows)

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [48]:
# fixing the date time dtype col
order_reviews_df['review_creation_date'] = order_reviews_df['review_creation_date'].astype('datetime64[ns]')
order_reviews_df['review_answer_timestamp'] = order_reviews_df['review_answer_timestamp'].astype('datetime64[ns]')

In [49]:
print(f"duplicate review id in table is {order_reviews_df['review_id'].duplicated().sum()}")
order_reviews_df = order_reviews_df.drop_duplicates(subset='review_id',keep='last')
# order_reviews_df.groupby('review_id')[['review_comment_title','review_comment_message']].value_counts().to_string('../arson/review_id.csv')
order_reviews_df.shape

duplicate review id in table is 814


(98410, 7)

In [50]:
print(order_reviews_df.isnull().sum())
## TO-DO: need to analysis these missing values and an approach to handle these values



review_id                      0
order_id                       0
review_score                   0
review_comment_title       86891
review_comment_message     57742
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [51]:
order_reviews_df[['review_id','review_comment_title','review_comment_message']].sample(20)
# checking whether

,review_id,review_comment_title,review_comment_message
68728,ccb91a1303d222e17a77aebb73e9c9d9,NaN,Chegou antes do prazo previsto
92471,6cfb54d8fe6568debd1e9c2e8f4d0b09,NaN,NaN
31630,eb06ad48cae0d1bb7854d21d6fdebafc,NaN,NaN
15536,4eb9ab08d52bd167c95099e6fe358263,NaN,NaN
20785,7dcbdbf12e770a00d43b0addc428959e,NaN,NaN
6562,6dfb0bef682d6dd4643aa5b5ae3432e4,NaN,Chegou antes do prazo.
66595,422c1afb7640cc680806495e97b042f1,NaN,Excelente produto a loja cumpriu rigorosamente...
37050,56df058093ebc328159047843a53f03e,NaN,NaN
97109,32662570b6c069849e11aeaeeab0063a,NaN,NaN
31133,5ab90f8ee51b8cc0072db4e29b949692,NaN,NaN


In [52]:

order_reviews_df[order_reviews_df['review_comment_message'].isna()][['review_id','review_comment_title','review_comment_message','review_score']]

,review_id,review_comment_title,review_comment_message,review_score
0,7bc2406110b926393aa56f80a40eba40,NaN,NaN,4
1,80e641a11e56f04c1ad469d5645fdfde,NaN,NaN,5
2,228ce5500dc1d8e020d8d1322874b6f0,NaN,NaN,5
5,15197aa66ff4d0650b5434f1b46cda19,NaN,NaN,1
6,07f9bee5d1b850860defd761afa7ff16,NaN,NaN,5
...,...,...,...,...
99217,c6b270c61f67c9f7cb07d84ea8aeaf8b,NaN,NaN,5
99218,af2dc0519de6e0720ef0c74292fb4114,NaN,NaN,5
99219,574ed12dd733e5fa530cfd4bbf39d7c9,NaN,NaN,5
99220,f3897127253a9592a73be9bdfdf4ed7a,NaN,NaN,5


In [53]:
## filling title and message with no comment
order_reviews_df[['review_comment_title','review_comment_message']] = order_reviews_df[['review_comment_title','review_comment_message']].fillna('No Comment')

In [54]:
# creating a new col has_commnet or not
order_reviews_df['has_comment'] = (order_reviews_df['review_comment_title'] != 'No Comments')

In [55]:
order_reviews_df.sample(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_comment
85778,4a3f767aa54f0bed484840d0c3326575,db47a089bc272d77b480a33e5286eaf5,5,No Comment,No Comment,2017-07-19,2017-07-19 23:28:52,True
22551,73dddbe2427b17e86e4e3938807ac35b,6244be66e1870c2392135aa8e2f61aee,5,No Comment,No Comment,2018-05-03,2018-05-07 16:28:58,True
71509,473eebaea742561740a227024af652ba,d5162716aed8b6a5ab095eb0a2940163,5,No Comment,No Comment,2017-07-04,2017-07-07 10:46:49,True
14586,14aeac040f36b8e1d53a934bf3508e8d,f203b250a442eac108f23c45f8b1efc8,5,No Comment,No Comment,2018-05-15,2018-05-16 11:27:33,True
57567,937804d7a75c25090020af23e32afe48,a4fa719430167664811c30680941ec40,5,No Comment,No Comment,2018-08-17,2018-08-19 23:17:40,True
91561,f487b4229b3412df53ea645c68768f73,95573e80e3966d4340d792f4f6c865b7,5,No Comment,No Comment,2018-01-19,2018-01-19 17:40:08,True
9929,83095b0e71b60ae410569c7919d3d14a,443b9c04691b94da8860a4b73512f94a,5,No Comment,No Comment,2017-10-21,2017-10-21 22:54:39,True
8022,4c9ff2fd5f6c682d311ce14f9e2d92b3,bfaa65d09b9fd34aa36a23ad90fa7db3,3,Última compra,Apesar do produto não vir ao identico da compr...,2018-06-29,2018-07-05 15:01:22,True
63420,dcb3c68821b178fc6733c5825819573b,0cf2af331f55382bd5698864658e73f0,1,No Comment,No Comment,2017-05-21,2017-05-22 14:08:37,True
33377,4abdfada09ad5b97d938d9fac2c487ba,1a1712253479ea7fd8c020c572ff78cc,4,No Comment,Bom vendedor,2017-07-27,2017-07-27 23:33:27,True


In [56]:
order_reviews_df.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
has_comment                0
dtype: int64

In [57]:
order_reviews_df[['review_comment_title','review_comment_message']].value_counts().to_string('../arson/comments.csv')

In [58]:
order_reviews_df.to_csv(clean_data_dir/'cleaned_order_reviews.csv')

In [59]:
order_items_df.sample(5)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
43262,6263f4308c3fde5104a2ead5e90dd41c,1,d47821b10559fffaefcf3e57d2b5ff76,0df3984f9dfb3d49ac6366acbd3bbb85,2017-07-04 09:55:08,189.90,35.13
96507,db06ef65b157c381358ccccdfa3d1aea,2,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-01-26 09:35:53,69.90,13.08
79052,b3d81ed08686a0fcfef9a9c3d5db6571,1,94254892148eb9bf4087a3e0886f3f30,7a67c85e85bb2ce8582c35f2203ad736,2017-06-26 11:15:10,59.99,17.67
112426,ff74e94493932bd63924387d576ed9b9,1,c0abb5707b6d57b4e7d9797222a77fc8,709e16e2b25c7474d980076c6bfc4806,2018-07-26 13:31:27,74.90,22.45
99005,e0a392ae82bc219375ace26ec32a6301,1,18f6ea7e8cff29f8053baac6762449f9,f80edd2c5aaa505cc4b0a3b219abf4b8,2018-06-14 02:54:00,54.90,11.15
